# ReelForge — One-Prompt AI Video Studio**Made for phone-first use. You do not need to write or edit code.**1. Tap **Connect** in the top-right corner.2. Open **Runtime → Run all**. Choose an available GPU runtime if Colab asks.3. When ReelForge opens, upload your raw clips/photos, optionally add your own audio, write one prompt, then tap **Create my video**.4. Choose **Fast preview**, **Balanced**, or **High detail**. Balanced is the default.ReelForge uses open model weights instead of paid generation APIs. Wan 2.1 is the primary text-to-video engine; AnimateDiff is the moving-video fallback. The renderer does **not** silently replace failed video generation with still images.The finished export is a vertical H.264/AAC MP4, with no added watermark. The pipeline can analyse uploaded frames, create AI B-roll, edit source media, add transitions/VFX, transcribe speech, generate a voice-over when appropriate, create an original instrumental score when no audio is supplied, and add transition sound effects.**Speed improvements:** balanced mode generates 65 frames per AI scene at 12 diffusion steps; high detail uses 81 frames and 16 steps. The Wan model stays loaded during the current Colab session, so a second render skips reloading it. The progress card reports scene and diffusion-step progress while a shot is being generated.The notebook runs in Google's Colab cloud runtime. Your media is uploaded to that runtime for processing; this repository stores the code, not your uploaded media. Free GPU access and session availability are controlled by Colab and are not guaranteed.Use media and music you own or have permission to use. AI scenes are short generated shots assembled into a complete edit, not a single continuous one-minute generated shot.

In [ ]:
#@title Install ReelForge's rendering stack (skip packages already present)
import importlib.metadata as metadata
import subprocess
import sys
import shutil
import os

# Colab usually already includes ffmpeg and several Python packages.
# Only install missing system tools so a normal launch avoids a full apt update.
apt_packages = []
if not shutil.which("ffmpeg"):
    apt_packages.append("ffmpeg")
if not shutil.which("espeak-ng"):
    apt_packages.append("espeak-ng")
if not os.path.exists("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"):
    apt_packages.append("fonts-dejavu-core")
if not os.path.exists("/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf"):
    apt_packages.append("fonts-noto-core")

if apt_packages:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *apt_packages], check=True)

# Install only missing/out-of-range dependencies. Torch is supplied by Colab.
from packaging.specifiers import SpecifierSet
from packaging.version import Version

requirements = [
    ("numpy", "numpy<2", "<2"),
    ("diffusers", "diffusers>=0.35.1,<0.37", ">=0.35.1,<0.37"),
    ("transformers", "transformers>=4.49.0,<5", ">=4.49.0,<5"),
    ("accelerate", "accelerate>=1.1.0", ">=1.1.0"),
    ("moviepy", "moviepy==1.0.3", "==1.0.3"),
    ("imageio", "imageio[ffmpeg]>=2.34", ">=2.34"),
    ("imageio-ffmpeg", "imageio-ffmpeg", ""),
    ("soundfile", "soundfile>=0.12.1", ">=0.12.1"),
    ("kokoro", "kokoro>=0.9.4", ">=0.9.4"),
    ("faster-whisper", "faster-whisper>=1.1.0", ">=1.1.0"),
    ("ipywidgets", "ipywidgets>=8.0", ">=8.0"),
    ("safetensors", "safetensors", ""),
    ("sentencepiece", "sentencepiece", ""),
    ("huggingface-hub", "huggingface_hub<1.0", "<1.0"),
    ("ftfy", "ftfy", ""),
]

to_install = []
for distribution, requirement, version_rule in requirements:
    try:
        installed = Version(metadata.version(distribution))
        if version_rule and installed not in SpecifierSet(version_rule):
            to_install.append(requirement)
    except metadata.PackageNotFoundError:
        to_install.append(requirement)

if to_install:
    print("Installing:", ", ".join(to_install))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *to_install],
        check=True
    )
else:
    print("Rendering dependencies already match. Skipping pip installs.")

print("ReelForge dependencies are ready. No paid API keys were used.")

In [ ]:
#@title Check that the Colab GPU is ready
import torch
from PIL import Image

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. Open Runtime → Change runtime type, choose an available GPU, reconnect, then use Runtime → Run all.")

if not hasattr(Image, "ANTIALIAS") and hasattr(Image, "Resampling"):
    Image.ANTIALIAS = Image.Resampling.LANCZOS

print("GPU ready:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

In [ ]:
#@title Launch ReelForge Studio
import urllib.request

REELFORGE_SCRIPT = "https://raw.githubusercontent.com/manjunathjadav360-rgb/Reel-forge-ai/main/colab/reelforge_cloud.py"
with urllib.request.urlopen(REELFORGE_SCRIPT, timeout=60) as response:
    source = response.read().decode("utf-8")

exec(compile(source, "reelforge_cloud.py", "exec"), globals())